In [ ]:
# Cell 1: List input files so we can confirm dataset paths
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
# Cell 2: Clean up any leftover staging dirs from previous runs
import os, shutil
staging_path = '/kaggle/working/upload_staging'
if os.path.exists(staging_path):
    shutil.rmtree(staging_path)
    print('Cleaned up staging area.')
else:
    print('No staging area to clean.')

In [ ]:
# Cell 3: Apply device-aware nn.Linear patch ONCE (prevents recursion)
import torch.nn as nn

if not hasattr(nn.Linear, '_is_patched'):
    _REAL_LINEAR_FWD = nn.Linear.forward

    def _device_aware_linear(self, x):
        if self.weight.device != x.device:
            self.weight = nn.Parameter(self.weight.to(x.device),
                                       requires_grad=self.weight.requires_grad)
        if self.bias is not None and self.bias.device != x.device:
            self.bias = nn.Parameter(self.bias.to(x.device),
                                     requires_grad=self.bias.requires_grad)
        return _REAL_LINEAR_FWD(self, x)

    nn.Linear.forward = _device_aware_linear
    nn.Linear._is_patched = True
    print('[patch] Device-aware nn.Linear applied.')
else:
    print('[patch] nn.Linear already patched — skipping.')

In [ ]:
# Cell 4: Locate the Janus repo and add it to sys.path
import os, sys

REPO = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train_avus_kaggle.py' in files and 'avus.py' in files:
        REPO = root
        break

if REPO is None:
    raise FileNotFoundError('Could not find train_avus_kaggle.py + avus.py in /kaggle/input')

print(f'Repo found at: {REPO}')
if REPO not in sys.path:
    sys.path.insert(0, REPO)
if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

In [ ]:
# Cell 5: Patch avus.py RMSNorm to be device-aware, write patched copy to /kaggle/working
import os, sys

REPO = [p for p in sys.path if os.path.exists(os.path.join(p, 'avus.py'))][0]
avus_src = open(os.path.join(REPO, 'avus.py')).read()

OLD_NORM = '        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n        return x * norm * self.weight'
NEW_NORM = ('        if self.weight.device != x.device:\n'
            '            self.weight = torch.nn.Parameter(self.weight.to(x.device), requires_grad=self.weight.requires_grad)\n'
            '        norm = torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)\n'
            '        return x * norm * self.weight')

if OLD_NORM in avus_src:
    avus_src = avus_src.replace(OLD_NORM, NEW_NORM)
    print('[patch] RMSNorm device-aware patch applied.')
else:
    print('[patch] RMSNorm pattern not found — already patched or different version.')

with open('/kaggle/working/avus.py', 'w') as f:
    f.write(avus_src)
print('[patch] Patched avus.py written to /kaggle/working/avus.py')

In [ ]:
# Cell 6: Set up Kaggle API credentials from secret
import os, json
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('Kiro')
    kaggle_dir = os.path.expanduser('~/.kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    creds = {'username': 'ishmaelsears', 'key': token}
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump(creds, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('[kaggle] API credentials configured from secret.')
except Exception as e:
    print(f'[kaggle] No Kiro secret ({e}) — auto-push to dataset disabled.')

In [ ]:
# Cell 7: Run training
# Reads train_avus_kaggle.py from repo, applies two patches, then exec()s it:
#   1. Forces KAGGLE_MODE = True
#   2. Fixes focal loss targets device mismatch (targets.to(logits.device))
import os, sys

REPO = [p for p in sys.path if os.path.exists(os.path.join(p, 'train_avus_kaggle.py'))][0]
script = open(os.path.join(REPO, 'train_avus_kaggle.py')).read()

# Patch 1: force Kaggle mode
script = script.replace(
    'KAGGLE_MODE           = False',
    'KAGGLE_MODE           = True'
)

# Patch 2: fix focal loss targets device
script = script.replace(
    '        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction="none")',
    '        targets = targets.to(logits.device)\n        ce   = F.cross_entropy(logits, targets, ignore_index=-1, reduction="none")'
)

print('[train] Patches applied. Starting training...')
print(f'[train] Script source: {os.path.join(REPO, "train_avus_kaggle.py")}')
exec(script)

In [ ]:
# Cell 8: Show output files and download links
import os
from IPython.display import FileLink, display

working = '/kaggle/working'
print('Output files:')
for f in sorted(os.listdir(working)):
    path = os.path.join(working, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / 1e6
        print(f'  {f}  ({size_mb:.1f} MB)')

# Download links for key outputs
for fname in ['avus_1b_weights.pt', 'skill_chart.png', 'skill_state.json', 'hbm_weights.pt']:
    fpath = os.path.join(working, fname)
    if os.path.exists(fpath):
        display(FileLink(fname))
    else:
        print(f'  (not found: {fname})')